# 04 — EOS model misspecification

In notebook 03 we performed hierarchical inference assuming that the EOS relation could be represented by the same polynomial order used to generate the mock data.

In this notebook we deliberately break that assumption.

The goal is to study **EOS model misspecification**.

A model is misspecified when the model family used in inference does not contain, or cannot accurately represent, the model that generated the data.

Here the physical object we want to infer is the shared mass--tidal-deformability relation

$$
\Lambda = \Lambda_{\rm EOS}(m).
$$

The mock data were generated using the EOS surrogate from notebook 00. In notebook 03 we inferred the EOS using the matched polynomial representation. In this notebook we repeat the hierarchical inference using different polynomial orders.

We compare:

- the matched inference result from notebook 03;
- an under-flexible model, for example a linear polynomial in $\log\Lambda(m)$;
- an over-flexible model, for example a higher-order polynomial.

The question is not only whether the inferred curve looks close to the truth. The deeper question is:

> If the data are noisy and limited, can hierarchical inference actually tell us which EOS model family is correct?

This notebook is intentionally left as an exercise. You should run it, inspect the diagnostic plots, and decide what the data are able, or unable, to distinguish.

## What you should learn

By the end of this notebook you should understand that:

1. Hierarchical inference does not automatically solve model misspecification.

2. A wrong but flexible or simple model can sometimes fit the data almost as well as the true model.

3. The ability to identify the correct EOS model depends on the number of events, their SNRs, and the precision of the tidal measurements.

4. Model comparison diagnostics such as log likelihood, DIC-like scores, and posterior predictive residuals are useful, but they are not magic.

5. If the data are weak, different EOS parameterizations may be practically indistinguishable.



## Colab setup

In [ ]:
# # Run this cell only on Colab.
# from google.colab import drive
# drive.mount('/content/drive')

# %cd /content/drive/MyDrive
# %cd ns-eos-population-tutorial

# !pip install -q numpyro corner

## Imports and paths

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import jax.scipy as jsp

import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_value

import arviz as az

from astropy.cosmology import Planck18 as cosmology
from astropy.constants import c

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "data/processed"
figures_dir = project_root / "figures"
results_dir = project_root / "results"

figures_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

C_KM_S = c.to("km/s").value

## Runtime choices


This notebook reruns the hierarchical inference several times, each time with a different polynomial order for the EOS model.

The list

```python
inference_eos_orders = [1, 7]
```

controls which misspecified EOS models are tested.

For example:

- order 1 corresponds to a linear model for $\log\Lambda(m)$;
- order 5 corresponds to the matched model used in notebook 03;
- order 7 corresponds to a more flexible model than the one used to generate the mock data.

The polynomial model is

$$
\log\Lambda_{\rm EOS}(m)
=
\sum_{k=0}^{K} a_k m^{K-k},
$$

where $K$ is the polynomial order.

Changing $K$ changes the function space available to the inference.

### Underfitting

If $K$ is too small, the model may be unable to represent the true EOS curve. This is called underfitting.

In this case the posterior may be biased because the model is forced to approximate the true relation with an overly rigid function.

### Overfitting or excessive flexibility

If $K$ is too large, the model may have enough freedom to fit noise or produce unphysical oscillations.

In this case the posterior may become broader, more prior dependent, or numerically less stable.

### Why this is an exercise

The correct conclusion is not known before running the notebook. Depending on the mock catalog realization, the linear model may be clearly worse, or it may be statistically hard to distinguish from the matched model.

This is the point of the exercise: with finite and noisy data, model misspecification may or may not be visible.

In [ ]:


# EOS orders to run in this notebook.
inference_eos_orders = [1, 7]

# These control runtime only. They do not change the generated catalog.
n_events_use = 20
n_pe_use = 500
n_injections_use = 2_000

# NumPyro settings.
num_warmup = 200
num_samples = 200
num_chains = 2
rng_seed = 123

# Optional hard cuts. Use these to test with/without the constraints.
apply_hard_eos_physical_prior = False
apply_hard_mc_variance_cut = False

# GWTC-style Monte Carlo likelihood variance cut.
max_log_likelihood_variance = 1.0


## Load data from previous notebooks

In [ ]:
# Tagged inputs from notebooks 00--02.
# Edit these filenames if you changed the upstream tags.
eos_fit_filename = "eos_fit__bsk24_m1p00_2p25_poly5.npz"
population_filename = "intrinsic_population__bsk24_m1p00_2p25_poly5__smoothbox_md_z20_n100000_seed31031993.npz"
detected_filename = "detected_events__bsk24_m1p00_2p25_poly5__smoothbox_md_z20_n100000_seed31031993__det_rho80_thr8_nev100_pe5000_seed1871.npz"
pe_filename = "mock_pe_samples__bsk24_m1p00_2p25_poly5__smoothbox_md_z20_n100000_seed31031993__det_rho80_thr8_nev100_pe5000_seed1871.npz"
injections_filename = "injections__bsk24_m1p00_2p25_poly5__smoothbox_md_z20_n100000_seed31031993__det_rho80_thr8_nev100_pe5000_seed1871__inj_n200000_z20_edge0p001_seed1871.npz"

eos_fit = np.load(processed_dir / eos_fit_filename)
intrinsic = np.load(processed_dir / population_filename)
detected = np.load(processed_dir / detected_filename)
pe = np.load(processed_dir / pe_filename)
injections = np.load(processed_dir / injections_filename)

def read_npz_string(data, key):
    value = data[key]
    if np.asarray(value).shape == ():
        return str(value.item())
    return str(value)

eos_tag = read_npz_string(eos_fit, "eos_tag")
eos_base_tag = read_npz_string(eos_fit, "eos_base_tag") if "eos_base_tag" in eos_fit else ""
fit_mass_tag = read_npz_string(eos_fit, "fit_mass_tag") if "fit_mass_tag" in eos_fit else ""
pop_tag = read_npz_string(intrinsic, "pop_tag")
detpe_tag = read_npz_string(detected, "detpe_tag")
inj_tag = read_npz_string(injections, "inj_tag")

n_events_available = detected["m1_det"].shape[0]
n_pe_available = pe["m1_det"].shape[1]
n_injections_available = injections["m1_det"].shape[0]
n_inj_drawn_total = int(injections["n_inj_drawn"])

if n_injections_use == -1:
    n_injections_use = n_injections_available

inf_tag = (
    f"inf_nev{n_events_use}_npe{n_pe_use}_ninj{n_injections_use}"
    f"_w{num_warmup}_s{num_samples}_c{num_chains}_seed{rng_seed}"
)
base_tag = f"{eos_tag}__{pop_tag}__{detpe_tag}__{inj_tag}"
matched_run_tag = f"{base_tag}__{inf_tag}"
matched_idata_filename = f"inferencedata__{matched_run_tag}.nc"

fit_min = float(eos_fit["fit_min"])
fit_max = float(eos_fit["fit_max"])
truth_eos_order = int(eos_fit["chosen_order"])
eos_coeffs_fit = np.array(eos_fit["chosen_coeffs"])
edge_pdf_value = float(intrinsic["edge_pdf_value"])

print(f"EOS tag: {eos_tag}")
print(f"Population tag: {pop_tag}")
print(f"Detection/PE tag: {detpe_tag}")
print(f"Injection tag: {inj_tag}")
print(f"Matched run tag expected from notebook 03: {matched_run_tag}")
print()
print(f"Available events: {n_events_available}")
print(f"Available PE samples per event: {n_pe_available}")
print(f"Available detected injections: {n_injections_available}")
print(f"Total injections drawn in notebook 02: {n_inj_drawn_total}")
print(f"Truth EOS order: {truth_eos_order}")
print("Truth EOS coefficients, highest power first:")
print(eos_coeffs_fit)

## Select a smaller dataset for fast runs

In [ ]:
rng = np.random.default_rng(rng_seed)

if n_events_use > n_events_available:
    raise ValueError(f"n_events_use={n_events_use} exceeds available events={n_events_available}")
if n_pe_use > n_pe_available:
    raise ValueError(f"n_pe_use={n_pe_use} exceeds available PE samples per event={n_pe_available}")
if n_injections_use > n_injections_available:
    raise ValueError(f"n_injections_use={n_injections_use} exceeds available detected injections={n_injections_available}")

event_idx = np.arange(n_events_available)[:n_events_use]
pe_idx = rng.choice(n_pe_available, size=n_pe_use, replace=False)
inj_idx = rng.choice(n_injections_available, size=n_injections_use, replace=False)

pe_m1_det = pe["m1_det"][event_idx][:, pe_idx]
pe_m2_det = pe["m2_det"][event_idx][:, pe_idx]
pe_d_l = pe["d_l"][event_idx][:, pe_idx]

obs_log_lambdatilde = pe["obs_log_lambdatilde"][event_idx]
sigma_log_lambdatilde = pe["sigma_log_lambdatilde"][event_idx]

true_m1 = detected["m1"][event_idx]
true_m2 = detected["m2"][event_idx]
true_mchirp = (true_m1 * true_m2) ** (3.0 / 5.0) / (true_m1 + true_m2) ** (1.0 / 5.0)

inj_m1_det = injections["m1_det"][inj_idx]
inj_m2_det = injections["m2_det"][inj_idx]
inj_d_l = injections["d_l"][inj_idx]
inj_log_p_draw = injections["log_p_draw"][inj_idx]

# We use only a random subset of found injections.  The selection estimator
# must therefore estimate the full found-injection sum by multiplying the
# subset sum by this factor.
n_inj_drawn = n_inj_drawn_total
inj_subset_factor = n_injections_available / n_injections_use

print(f"Using events: {n_events_use}")
print(f"Using PE samples per event: {n_pe_use}")
print(f"Using detected injections: {n_injections_use}")
print(f"Detected-injection subset factor: {inj_subset_factor:.6g}")
print(f"Total injections drawn in notebook 02: {n_inj_drawn}")

## Matched model from notebook 03

We load the result from notebook 03 as the reference inference.

The matched model is not necessarily perfect in an absolute sense, but it uses the same polynomial order as the surrogate used to generate the mock data.

This gives us a baseline against which we compare the misspecified models.

The comparison is between different inference assumptions applied to the same mock data set:

$$
\text{same events}
+
\text{same PE samples}
+
\text{same injections}
+
\text{different EOS model families}.
$$

This is important. If the results differ, the difference is due to the EOS model used in the inference, not to a different simulated population or a different noise realization.

In [ ]:
matched_idata_path = results_dir / matched_idata_filename
matched_idata = az.from_netcdf(matched_idata_path)

print(f"Loaded matched inference result from {matched_idata_path}")

## Cosmology grid

In [ ]:
z_grid_np = np.linspace(1e-6, 25.0, 6000)
d_l_grid_np = cosmology.luminosity_distance(z_grid_np).value
dc_grid_np = cosmology.comoving_distance(z_grid_np).value
Hz_grid_np = cosmology.H(z_grid_np).value

log_dV_dz_grid_np = np.log(4.0 * np.pi) + np.log(C_KM_S) - np.log(Hz_grid_np) + 2.0 * np.log(dc_grid_np)
ddL_dz_grid_np = dc_grid_np + (1.0 + z_grid_np) * C_KM_S / Hz_grid_np
log_ddL_dz_grid_np = np.log(ddL_dz_grid_np)

z_grid = jnp.asarray(z_grid_np)
d_l_grid = jnp.asarray(d_l_grid_np)
log_dV_dz_grid = jnp.asarray(log_dV_dz_grid_np)
log_ddL_dz_grid = jnp.asarray(log_ddL_dz_grid_np)

def z_from_d_l_jax(d_l):
    return jnp.interp(d_l, d_l_grid, z_grid)

def interp_log_dV_dz(z):
    return jnp.interp(z, z_grid, log_dV_dz_grid)

def interp_log_ddL_dz(z):
    return jnp.interp(z, z_grid, log_ddL_dz_grid)


## Population model

## What stays fixed across the comparisons?

The population model is kept structurally the same as in notebook 03.

The source-frame population is still

$$
p_{\rm pop}^{\rm src}(m_1,m_2,z|\lambda)
=
2p(m_1|\lambda)p(m_2|\lambda)p(z|\lambda),
\qquad
m_1\geq m_2.
$$

The redshift model is still

$$
p(z|\lambda)
\propto
\frac{R(z|\lambda)}{1+z}
\frac{dV_c}{dz}.
$$

The detector-frame density is still obtained using the Jacobian

$$
p_{\rm pop}^{\rm det}
=
p_{\rm pop}^{\rm src}
\frac{1}{(1+z)^2}
\left(
\frac{dd_L}{dz}
\right)^{-1}.
$$

The selection correction is still estimated using the same detected injections.

The event likelihood is still computed by PE-sample reweighting.

The only conceptual change is the family of EOS curves allowed in the inference:

$$
\Lambda_{\rm EOS}(m;\phi_{\rm EOS})
\quad
\longrightarrow
\quad
\Lambda_{\rm EOS}^{(K)}(m;\phi_{\rm EOS}^{(K)}).
$$

This isolation is important: it lets us study EOS model misspecification directly.

In [ ]:
mass_grid = jnp.linspace(fit_min, fit_max, 1000)

def sigmoid_jax(x):
    return jax.nn.sigmoid(x)

def bounded_smooth_mass_unnorm(m, m_min, m_max, edge_width):
    logit_edge = jnp.log((1.0 - edge_pdf_value) / edge_pdf_value)
    edge_scale = edge_width / logit_edge
    left = sigmoid_jax((m - m_min - edge_width) / edge_scale)
    right = sigmoid_jax((m_max - edge_width - m) / edge_scale)
    return left * right

def log_mass_pdf(m, m_min, m_max, edge_width):
    pdf_grid = bounded_smooth_mass_unnorm(mass_grid, m_min, m_max, edge_width)
    norm = jnp.trapezoid(pdf_grid, mass_grid)

    pdf = bounded_smooth_mass_unnorm(m, m_min, m_max, edge_width) / norm

    in_support = (m >= fit_min) & (m <= fit_max)
    return jnp.where(
        in_support,
        jnp.log(jnp.maximum(pdf, 1e-300)),
        -1e30,
    )

def merger_rate_density_jax(z, alpha_z, beta_z, z_p):
    numerator = (1.0 + z) ** alpha_z
    denominator = 1.0 + ((1.0 + z) / (1.0 + z_p)) ** (alpha_z + beta_z)
    return numerator / denominator

def log_redshift_pdf(z, alpha_z, beta_z, z_p):
    rate_grid = merger_rate_density_jax(z_grid, alpha_z, beta_z, z_p)
    integrand_grid = rate_grid * jnp.exp(log_dV_dz_grid) / (1.0 + z_grid)
    norm = jnp.trapezoid(integrand_grid, z_grid)

    log_rate = jnp.log(merger_rate_density_jax(z, alpha_z, beta_z, z_p))
    return log_rate + interp_log_dV_dz(z) - jnp.log1p(z) - jnp.log(norm)

def log_pop_source(m1, m2, z, m_min, m_max, edge_width, alpha_z, beta_z, z_p):
    return (
        jnp.log(2.0)
        + log_mass_pdf(m1, m_min, m_max, edge_width)
        + log_mass_pdf(m2, m_min, m_max, edge_width)
        + log_redshift_pdf(z, alpha_z, beta_z, z_p)
    )


## EOS model and deterministic tidal constraint

For each polynomial order $K$, the inference model assumes

$$
\log\Lambda_{\rm EOS}^{(K)}(m)
=
\sum_{k=0}^{K} a_k m^{K-k}.
$$

The coefficients

$$
\phi_{\rm EOS}^{(K)}=(a_0,\ldots,a_K)
$$

are inferred from the data.

For a proposed EOS curve and source-frame masses, the model predicts

$$
\Lambda_1
=
\Lambda_{\rm EOS}^{(K)}(m_1;\phi_{\rm EOS}^{(K)}),
\qquad
\Lambda_2
=
\Lambda_{\rm EOS}^{(K)}(m_2;\phi_{\rm EOS}^{(K)}).
$$

The corresponding binary tidal parameter is

$$
\tilde{\Lambda}_{\rm EOS}^{(K)}
=
\frac{16}{13}
\frac{
(m_1+12m_2)m_1^4\Lambda_1
+
(m_2+12m_1)m_2^4\Lambda_2
}
{(m_1+m_2)^5}.
$$

The deterministic EOS assumption means that the population model contains

$$
p(\tilde{\Lambda}|m_1,m_2,\phi_{\rm EOS}^{(K)})
=
\delta\left[
\tilde{\Lambda}
-
\tilde{\Lambda}_{\rm EOS}^{(K)}(m_1,m_2;\phi_{\rm EOS}^{(K)})
\right].
$$

As in notebook 03, the tidal part of the mock likelihood is evaluated at the EOS prediction,

$$
p(d_i^{\rm tidal}|m_1,m_2,\phi_{\rm EOS}^{(K)})
=
\mathcal{N}
\left[
\widehat{\log\tilde{\Lambda}}_i
\middle|
\log\tilde{\Lambda}_{\rm EOS}^{(K)}(m_1,m_2;\phi_{\rm EOS}^{(K)}),
\sigma_{\log\tilde{\Lambda},i}
\right].
$$

The key difference from notebook 03 is that the function family itself may now be wrong.

If $K$ is too small, the true relation may not be representable. If $K$ is too large, the model may represent the truth but also many unnecessary shapes.

This is the model-selection problem.

In [ ]:
def polyval_jax(coeffs, x):
    y = jnp.zeros_like(x)
    for c in coeffs:
        y = y * x + c
    return y

def lambda_of_mass_jax(m, coeffs):
    return jnp.exp(polyval_jax(coeffs, m))

def lambda_tilde_jax(m1, m2, lambda1, lambda2):
    return (16.0 / 13.0) * (
        (m1 + 12.0 * m2) * m1**4 * lambda1
        + (m2 + 12.0 * m1) * m2**4 * lambda2
    ) / (m1 + m2) ** 5

def log_tidal_likelihood(log_lambda_tilde_eos, obs_log_lambdatilde, sigma):
    return dist.Normal(obs_log_lambdatilde[:, None], sigma[:, None]).log_prob(log_lambda_tilde_eos)


## Parameter configuration

## Priors for different EOS orders

For each polynomial order, the notebook constructs a separate set of EOS coefficients.

An order-$K$ polynomial has $K+1$ coefficients,

$$
\phi_{\rm EOS}^{(K)}=(a_0,\ldots,a_K).
$$

Therefore, changing the polynomial order changes both:

1. the number of EOS parameters;
2. the set of functions that can be represented.

This is why comparing different polynomial orders is not just comparing different parameter values. It is comparing different model families.

The priors on the coefficients are chosen to keep the inference numerically stable and to avoid obviously pathological EOS curves. However, these priors are part of the model. A different prior on polynomial coefficients could lead to a different preference between models.

This is one reason why model comparison in EOS inference must be interpreted carefully.

In [ ]:
m_min_true = float(intrinsic["m_min_true"])
m_max_true = float(intrinsic["m_max_true"])
edge_width_true = float(intrinsic["edge_width_true"])
alpha_z_true = float(intrinsic["alpha_z"])
beta_z_true = float(intrinsic["beta_z"])
z_p_true = float(intrinsic["z_p"])

def fit_truth_coeffs_to_order(order):
    m = np.linspace(fit_min, fit_max, 500)
    log_lambda_truth = np.polyval(eos_coeffs_fit, m)
    return np.polyfit(m, log_lambda_truth, order)

def make_param_config(inference_eos_order):
    coeff_center = fit_truth_coeffs_to_order(inference_eos_order)

    cfg = {
        "m_min": {
            "sample": True,
            "fixed": m_min_true,
            "prior": ("uniform", fit_min - 0.05, fit_min + 0.05),
            "init": m_min_true,
        },
        "m_max": {
            "sample": True,
            "fixed": m_max_true,
            "prior": ("uniform", fit_max - 0.05, fit_max + 0.05),
            "init": m_max_true,
        },
        "edge_width": {
            "sample": False,
            "fixed": edge_width_true,
            "prior": ("uniform", 0.01, 0.15),
            "init": edge_width_true,
        },
        "alpha_z": {
            "sample": True,
            "fixed": alpha_z_true,
            "prior": ("uniform", 1.0, 5.0),
            "init": alpha_z_true,
        },
        "beta_z": {
            "sample": True,
            "fixed": beta_z_true,
            "prior": ("uniform", 1.0, 5.0),
            "init": beta_z_true,
        },
        "z_p": {
            "sample": True,
            "fixed": z_p_true,
            "prior": ("uniform", 1.0, 3.0),
            "init": z_p_true,
        },
    }

    for k, coeff in enumerate(coeff_center):
        width = 0.02 * abs(float(coeff)) + 0.01
        cfg[f"eos_a{k}"] = {
            "sample": True,
            "fixed": float(coeff),
            "prior": ("normal", float(coeff), width),
            "init": float(coeff),
        }

    return cfg, coeff_center

## Rate-marginalized likelihood

## Hierarchical likelihood used for each EOS model

For every EOS order $K$, we use the same rate-marginalized hierarchical likelihood.

For event $i$,

$$
\widehat{p}(d_i|\lambda,K)
=
\frac{1}{N_{\rm PE}}
\sum_{j=1}^{N_{\rm PE}}
\frac{
p_{\rm pop}^{\rm det}(x_{ij}|\lambda,K)
}{
\pi_{\rm PE}(x_{ij})
}
p(d_i^{\rm tidal}|x_{ij},\phi_{\rm EOS}^{(K)}).
$$

Here

$$
x_{ij}=(m_{1,z,ij},m_{2,z,ij},d_{L,ij})
$$

is PE sample $j$ for event $i$.

The tidal likelihood is

$$
p(d_i^{\rm tidal}|x_{ij},\phi_{\rm EOS}^{(K)})
=
\mathcal{N}
\left[
\widehat{\log\tilde{\Lambda}}_i
\middle|
\log\tilde{\Lambda}_{\rm EOS}^{(K)}(x_{ij};\phi_{\rm EOS}^{(K)}),
\sigma_{\log\tilde{\Lambda},i}
\right].
$$

The selection efficiency for model $K$ is estimated with the same injection set,

$$
\widehat{\xi}(\lambda,K)
=
\frac{1}{N_{\rm draw}}
\sum_{j\in{\rm found}}
\frac{
p_{\rm pop}^{\rm det}(\theta_j|\lambda,K)
}{
p_{\rm draw}^{\rm det}(\theta_j)
}.
$$

The total likelihood is

$$
\log\widehat{\mathcal{L}}(\lambda,K)
=
\sum_i
\log\widehat{p}(d_i|\lambda,K)
-
N_{\rm det}\log\widehat{\xi}(\lambda,K).
$$

This is the same likelihood structure as notebook 03. The only change is the EOS function family.

In [ ]:
pe_m1_det_j = jnp.asarray(pe_m1_det)
pe_m2_det_j = jnp.asarray(pe_m2_det)
pe_d_l_j = jnp.asarray(pe_d_l)

obs_log_lambdatilde_j = jnp.asarray(obs_log_lambdatilde)
sigma_log_lambdatilde_j = jnp.asarray(sigma_log_lambdatilde)

inj_m1_det_j = jnp.asarray(inj_m1_det)
inj_m2_det_j = jnp.asarray(inj_m2_det)
inj_d_l_j = jnp.asarray(inj_d_l)
inj_log_p_draw_j = jnp.asarray(inj_log_p_draw)

def sample_parameter(name, cfg):
    prior = cfg["prior"]
    if cfg["sample"]:
        if prior[0] == "uniform":
            return numpyro.sample(name, dist.Uniform(prior[1], prior[2]))
        if prior[0] == "normal":
            return numpyro.sample(name, dist.Normal(prior[1], prior[2]))
        raise ValueError(f"Unknown prior type for {name}: {prior[0]}")
    return jnp.asarray(cfg["fixed"])

def relative_variance_from_logweights(logw, axis):
    max_logw = jnp.max(logw, axis=axis, keepdims=True)
    w = jnp.exp(logw - max_logw)

    n = logw.shape[axis]
    mean_w = jnp.mean(w, axis=axis, keepdims=True)
    var_mean = jnp.sum((w - mean_w) ** 2, axis=axis) / (n * (n - 1))
    mean_w_flat = jnp.squeeze(mean_w, axis=axis)

    return var_mean / (mean_w_flat ** 2)

def logmeanexp(logw, axis):
    return jsp.special.logsumexp(logw, axis=axis) - jnp.log(logw.shape[axis])



def soft_barrier_lower(x, xmin, width):
    return -jax.nn.softplus((xmin - x) / width)

def soft_barrier_upper(x, xmax, width):
    return -jax.nn.softplus((x - xmax) / width)

    

@jax.jit
def compute_hierarchical_loglike(
    m_min,
    m_max,
    edge_width,
    alpha_z,
    beta_z,
    z_p,
    coeffs,
):
    pe_z = z_from_d_l_jax(pe_d_l_j)
    pe_m1 = pe_m1_det_j / (1.0 + pe_z)
    pe_m2 = pe_m2_det_j / (1.0 + pe_z)

    pe_log_pop_src = log_pop_source(
        pe_m1, pe_m2, pe_z,
        m_min, m_max, edge_width,
        alpha_z, beta_z, z_p,
    )

    pe_log_jac = -2.0 * jnp.log1p(pe_z) - interp_log_ddL_dz(pe_z)
    pe_log_pe_prior = 2.0 * jnp.log(pe_d_l_j)

    pe_lambda1 = lambda_of_mass_jax(pe_m1, coeffs)
    pe_lambda2 = lambda_of_mass_jax(pe_m2, coeffs)
    pe_lambda_tilde_eos = lambda_tilde_jax(pe_m1, pe_m2, pe_lambda1, pe_lambda2)
    pe_log_lambda_tilde_eos = jnp.log(jnp.maximum(pe_lambda_tilde_eos, 1e-300))

    logL_tidal = log_tidal_likelihood(
        pe_log_lambda_tilde_eos,
        obs_log_lambdatilde_j,
        sigma_log_lambdatilde_j,
    )

    log_event_weights = pe_log_pop_src + pe_log_jac - pe_log_pe_prior + logL_tidal
    log_L_events = logmeanexp(log_event_weights, axis=1)
    event_rel_var = relative_variance_from_logweights(log_event_weights, axis=1)

    inj_z = z_from_d_l_jax(inj_d_l_j)
    inj_m1 = inj_m1_det_j / (1.0 + inj_z)
    inj_m2 = inj_m2_det_j / (1.0 + inj_z)

    inj_log_pop_src = log_pop_source(
        inj_m1, inj_m2, inj_z,
        m_min, m_max, edge_width,
        alpha_z, beta_z, z_p,
    )
    inj_log_jac = -2.0 * jnp.log1p(inj_z) - interp_log_ddL_dz(inj_z)
    inj_log_pop_det = inj_log_pop_src + inj_log_jac

    log_inj_weights = inj_log_pop_det - inj_log_p_draw_j

    max_logW = jnp.max(log_inj_weights)
    W_scaled = jnp.exp(log_inj_weights - max_logW)

    sum_W_scaled = jnp.sum(W_scaled)
    sum_W2_scaled = jnp.sum(W_scaled ** 2)

    # Correct for using a random subset of the found injections.
    # The full found-injection sum is estimated as
    # inj_subset_factor times the subset sum.
    xi_scaled = inj_subset_factor * sum_W_scaled / n_inj_drawn
    log_xi = max_logW + jnp.log(inj_subset_factor * sum_W_scaled) - jnp.log(n_inj_drawn)

    # Same correction for the second moment entering the GWTC-style MC variance.
    var_xi_scaled = (
        inj_subset_factor * sum_W2_scaled - n_inj_drawn * xi_scaled ** 2
    ) / (n_inj_drawn * (n_inj_drawn - 1))
    var_xi_scaled = jnp.maximum(var_xi_scaled, 0.0)
    xi_rel_var = var_xi_scaled / (xi_scaled ** 2)

    n_det = pe_m1_det_j.shape[0]
    log_likelihood_variance = jnp.sum(event_rel_var) + n_det**2 * xi_rel_var

    log_likelihood = jnp.sum(log_L_events) - n_det * log_xi

    return (
        log_likelihood,
        log_xi,
        xi_rel_var,
        jnp.sum(event_rel_var),
        log_likelihood_variance,
    )



def hierarchical_model():
    m_min = sample_parameter("m_min", param_config["m_min"])
    m_max = sample_parameter("m_max", param_config["m_max"])
    edge_width = sample_parameter("edge_width", param_config["edge_width"])
    alpha_z = sample_parameter("alpha_z", param_config["alpha_z"])
    beta_z = sample_parameter("beta_z", param_config["beta_z"])
    z_p = sample_parameter("z_p", param_config["z_p"])

    coeffs = []
    for k in range(eos_order + 1):
        coeffs.append(sample_parameter(f"eos_a{k}", param_config[f"eos_a{k}"]))
    coeffs = jnp.stack(coeffs)

    numpyro.factor("mass_order_constraint", jnp.where(m_min < m_max, 0.0, -jnp.inf))

    log_lambda_grid = polyval_jax(coeffs, mass_grid)

    # quick version - makes divergences appear
    eos_valid = (
        jnp.all(jnp.isfinite(log_lambda_grid))
        & jnp.all(log_lambda_grid > jnp.log(0.1))
        & jnp.all(log_lambda_grid < jnp.log(1e5))
        & jnp.all(jnp.diff(log_lambda_grid) < 0.0)
    )
    
    numpyro.deterministic("eos_is_physical", eos_valid)
    if apply_hard_eos_physical_prior:
        numpyro.factor("eos_physical_prior", jnp.where(eos_valid, 0.0, -1e30))

    # lengty version - smooths boundaries
    # log_lambda_min = jnp.log(0.1)
    # log_lambda_max = jnp.log(1e5)
    
    # eos_barrier_width = 0.05
    
    # log_lambda_lower_penalty = jnp.sum(
    #     soft_barrier_lower(log_lambda_grid, log_lambda_min, eos_barrier_width)
    # )
    
    # log_lambda_upper_penalty = jnp.sum(
    #     soft_barrier_upper(log_lambda_grid, log_lambda_max, eos_barrier_width)
    # )
    
    # monotonicity_penalty = jnp.sum(
    #     -jax.nn.softplus(jnp.diff(log_lambda_grid) / eos_barrier_width)
    # )
    
    # numpyro.factor(
    #     "eos_physical_prior",
    #     log_lambda_lower_penalty
    #     + log_lambda_upper_penalty
    #     + monotonicity_penalty,
    # )

    (
        log_likelihood,
        log_xi,
        selection_rel_var,
        sum_event_rel_var,
        log_likelihood_variance,
    ) = compute_hierarchical_loglike(
        m_min,
        m_max,
        edge_width,
        alpha_z,
        beta_z,
        z_p,
        coeffs,
        )

    numpyro.deterministic("log_xi", log_xi)
    numpyro.deterministic("selection_rel_var", selection_rel_var)
    numpyro.deterministic("sum_event_rel_var", sum_event_rel_var)
    numpyro.deterministic("log_likelihood_variance", log_likelihood_variance)
    numpyro.deterministic("log_likelihood_value", log_likelihood)

    numpyro.deterministic(
        "passes_mc_variance_cut",
        log_likelihood_variance < max_log_likelihood_variance,
        )

    if apply_hard_mc_variance_cut:
        numpyro.factor("variance_cut", jnp.where(log_likelihood_variance < max_log_likelihood_variance, 0.0, -1e30))

    # mc_variance_barrier_width = 0.1

    # mc_variance_penalty = -jax.nn.softplus(
    #     (log_likelihood_variance - max_log_likelihood_variance)
    #     / mc_variance_barrier_width
    # )
    
    # numpyro.factor("mc_variance_cut", mc_variance_penalty)

    numpyro.factor("hierarchical_likelihood", log_likelihood)

## Exercise: run the misspecified EOS inferences

This cell runs the hierarchical inference for each EOS order listed in

```python
inference_eos_orders
```

You should treat this as the main exercise of the notebook.

For each model, ask:

1. Does the sampler run successfully?

2. Does the inferred EOS curve recover the true curve?

3. Is the posterior band wider or narrower than in the matched model?

4. Does the model give a visibly biased EOS relation?

5. Are the likelihood diagnostics significantly worse than the matched model?

6. Are the differences large enough that you would confidently reject one model?

The important point is that visual bias and statistical distinguishability are not the same thing.

A model can be visibly different from the truth in some mass range but still fit the detected events well if the data contain little information in that region.

Conversely, a model can have a posterior band that includes the truth but still be disfavored if it wastes prior volume or fits the events less efficiently.

In [ ]:
def sample_parameter(name, cfg):
    prior = cfg["prior"]
    if cfg["sample"]:
        if prior[0] == "uniform":
            return numpyro.sample(name, dist.Uniform(prior[1], prior[2]))
        if prior[0] == "normal":
            return numpyro.sample(name, dist.Normal(prior[1], prior[2]))
        raise ValueError(prior[0])
    return cfg["fixed"]

def json_safe(x):
    if isinstance(x, (str, int, float, bool)) or x is None:
        return x
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, np.generic):
        return x.item()
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, tuple):
        return [json_safe(v) for v in x]
    if isinstance(x, list):
        return [json_safe(v) for v in x]
    if isinstance(x, dict):
        return {str(k): json_safe(v) for k, v in x.items()}
    return str(x)

def run_inference_for_eos_order(inference_eos_order, model_label):
    param_config, coeff_center = make_param_config(inference_eos_order)
    eos_model_tag = f"eosmodel_poly{inference_eos_order}"
    run_tag = f"{base_tag}__{eos_model_tag}__{inf_tag}"

    def hierarchical_model():
        m_min = sample_parameter("m_min", param_config["m_min"])
        m_max = sample_parameter("m_max", param_config["m_max"])
        edge_width = sample_parameter("edge_width", param_config["edge_width"])
        alpha_z = sample_parameter("alpha_z", param_config["alpha_z"])
        beta_z = sample_parameter("beta_z", param_config["beta_z"])
        z_p = sample_parameter("z_p", param_config["z_p"])

        coeffs = []
        for k in range(inference_eos_order + 1):
            coeffs.append(sample_parameter(f"eos_a{k}", param_config[f"eos_a{k}"]))
        coeffs = jnp.stack(coeffs)

        numpyro.factor(
            "mass_order_constraint",
            jnp.where(m_min < m_max, 0.0, -1e30),
        )

        log_lambda_grid = polyval_jax(coeffs, mass_grid)
        eos_valid = (
            jnp.all(jnp.isfinite(log_lambda_grid))
            & jnp.all(log_lambda_grid > jnp.log(0.1))
            & jnp.all(log_lambda_grid < jnp.log(1e5))
            & jnp.all(jnp.diff(log_lambda_grid) < 0.0)
        )
        numpyro.deterministic("eos_physical_valid", eos_valid)
        if apply_hard_eos_physical_prior:
            numpyro.factor("eos_physical_prior", jnp.where(eos_valid, 0.0, -1e30))

        (
            log_likelihood,
            log_xi,
            selection_rel_var,
            sum_event_rel_var,
            log_likelihood_variance,
        ) = compute_hierarchical_loglike(
            m_min,
            m_max,
            edge_width,
            alpha_z,
            beta_z,
            z_p,
            coeffs,
        )

        numpyro.deterministic("log_xi", log_xi)
        numpyro.deterministic("selection_rel_var", selection_rel_var)
        numpyro.deterministic("sum_event_rel_var", sum_event_rel_var)
        numpyro.deterministic("log_likelihood_variance", log_likelihood_variance)
        numpyro.deterministic("log_likelihood_value", log_likelihood)
        numpyro.deterministic(
            "passes_mc_variance_cut",
            log_likelihood_variance < max_log_likelihood_variance,
        )

        if apply_hard_mc_variance_cut:
            numpyro.factor(
                "variance_cut",
                jnp.where(log_likelihood_variance < max_log_likelihood_variance, 0.0, -1e30),
            )

        numpyro.factor("hierarchical_likelihood", log_likelihood)

    init_values = {
        name: cfg["init"]
        for name, cfg in param_config.items()
        if cfg["sample"]
    }

    nuts = NUTS(
        hierarchical_model,
        init_strategy=init_to_value(values=init_values),
        target_accept_prob=0.85,
    )
    mcmc = MCMC(
        nuts,
        num_warmup=num_warmup,
        num_samples=num_samples,
        num_chains=num_chains,
    )

    print(f"Running {model_label}: EOS order {inference_eos_order}")
    mcmc.run(jax.random.PRNGKey(rng_seed + inference_eos_order))
    mcmc.print_summary()

    idata = az.from_numpyro(mcmc)
    summary = az.summary(idata, round_to=3)

    summary_path = results_dir / f"summary__{run_tag}.csv"
    idata_path = results_dir / f"inferencedata__{run_tag}.nc"
    posterior_path = results_dir / f"posterior_samples__{run_tag}.npz"
    config_path = results_dir / f"config__{run_tag}.json"

    summary.to_csv(summary_path)
    idata.to_netcdf(idata_path)

    posterior_flat = mcmc.get_samples(group_by_chain=False)
    np.savez(
        posterior_path,
        **{name: np.asarray(value) for name, value in posterior_flat.items()},
    )

    run_config = {
        "run_tag": run_tag,
        "base_tag": base_tag,
        "eos_model_tag": eos_model_tag,
        "model_label": model_label,
        "inference_eos_order": inference_eos_order,
        "truth_eos_order": truth_eos_order,
        "eos_tag": eos_tag,
        "pop_tag": pop_tag,
        "detpe_tag": detpe_tag,
        "inj_tag": inj_tag,
        "inf_tag": inf_tag,
        "input_files": {
            "eos_fit_filename": eos_fit_filename,
            "population_filename": population_filename,
            "detected_filename": detected_filename,
            "pe_filename": pe_filename,
            "injections_filename": injections_filename,
            "matched_idata_filename": matched_idata_filename,
        },
        "runtime": {
            "n_events_use": n_events_use,
            "n_pe_use": n_pe_use,
            "n_injections_use": n_injections_use,
            "n_injections_available": n_injections_available,
            "inj_subset_factor": inj_subset_factor,
            "n_inj_drawn": n_inj_drawn,
            "num_warmup": num_warmup,
            "num_samples": num_samples,
            "num_chains": num_chains,
            "rng_seed": rng_seed,
            "apply_hard_eos_physical_prior": apply_hard_eos_physical_prior,
            "apply_hard_mc_variance_cut": apply_hard_mc_variance_cut,
            "max_log_likelihood_variance": max_log_likelihood_variance,
        },
        "truth": {
            "fit_min": fit_min,
            "fit_max": fit_max,
            "eos_order": truth_eos_order,
            "eos_coeffs_fit": eos_coeffs_fit,
            "m_min_true": m_min_true,
            "m_max_true": m_max_true,
            "edge_width_true": edge_width_true,
            "alpha_z_true": alpha_z_true,
            "beta_z_true": beta_z_true,
            "z_p_true": z_p_true,
        },
        "param_config": param_config,
    }

    with open(config_path, "w") as f:
        json.dump(json_safe(run_config), f, indent=2)

    trace_var_names = [
        name
        for name in idata.posterior.data_vars
        if idata.posterior[name].dtype.kind not in ["b"]
    ]
    az.plot_trace(idata, var_names=trace_var_names, compact=True)
    plt.tight_layout()
    trace_path = figures_dir / f"trace__{run_tag}.png"
    plt.savefig(trace_path, dpi=200, bbox_inches="tight")
    plt.show()

    return {
        "label": model_label,
        "order": inference_eos_order,
        "run_tag": run_tag,
        "idata": idata,
        "param_config": param_config,
        "coeff_center": coeff_center,
        "summary": summary,
    }

In [ ]:
misspec_results = {}

for order in inference_eos_orders:
    label = f"Poly {order} inference"
    misspec_results[label] = run_inference_for_eos_order(order, label)

print("Completed misspecified EOS runs:")
for label, result in misspec_results.items():
    print(f"  {label}: {result['run_tag']}")

## Interpreting the outputs

The notebook now has several posterior samples:

- the matched model from notebook 03;
- one or more misspecified EOS models from this notebook.

All of them are conditioned on the same detected events and the same injection set.

The rest of the notebook compares the models in several ways:

1. EOS posterior bands in function space.

2. Bias in $\log\Lambda(m)$ as a function of mass.

3. Posterior mass-distribution checks.

4. Tidal residuals for the detected events.

5. Summary metrics such as median log likelihood and DIC-like scores.

No single diagnostic is definitive. The goal is to build a consistent picture.

## Collect matched and misspecified results

In [ ]:
def flatten_idata_posterior(idata):
    return {
        name: np.asarray(
            idata.posterior[name]
            .stack(sample=("chain", "draw"))
            .transpose("sample", ...)
            .values
        )
        for name in idata.posterior.data_vars
    }

all_results = {
    f"Matched poly {truth_eos_order}": {
        "label": f"Matched poly {truth_eos_order}",
        "order": truth_eos_order,
        "run_tag": matched_run_tag,
        "idata": matched_idata,
        "param_config": None,
    }
}
all_results.update(misspec_results)

for result in all_results.values():
    result["posterior"] = flatten_idata_posterior(result["idata"])

comparison_tag = f"{base_tag}__eosmisspec__{inf_tag}"
print(f"Comparison tag: {comparison_tag}")

## Helper functions for comparison diagnostics

In [ ]:
def get_coeff_draws(posterior, order):
    return np.column_stack([
        np.asarray(posterior[f"eos_a{k}"])
        for k in range(order + 1)
    ])

def eos_loglambda_draws(posterior, order, m_grid):
    coeff_draws = get_coeff_draws(posterior, order)
    return np.array([
        np.polyval(coeff_draws[i], m_grid)
        for i in range(coeff_draws.shape[0])
    ])

def result_band(y_draws):
    return (
        np.percentile(y_draws, 5, axis=0),
        np.percentile(y_draws, 50, axis=0),
        np.percentile(y_draws, 95, axis=0),
    )

def bounded_smooth_mass_unnorm_np(m, m_min, m_max, edge_width, edge_pdf_value):
    logit_edge = np.log((1.0 - edge_pdf_value) / edge_pdf_value)
    edge_scale = edge_width / logit_edge
    left = 1.0 / (1.0 + np.exp(-(m - m_min - edge_width) / edge_scale))
    right = 1.0 / (1.0 + np.exp(-(m_max - edge_width - m) / edge_scale))
    return left * right

def mass_pdf_draws_from_posterior(posterior, m_grid):
    n_draws = len(next(iter(posterior.values())))
    pdf_draws = []
    for draw_id in range(n_draws):
        m_min_draw = float(posterior["m_min"][draw_id]) if "m_min" in posterior else m_min_true
        m_max_draw = float(posterior["m_max"][draw_id]) if "m_max" in posterior else m_max_true
        edge_width_draw = float(posterior["edge_width"][draw_id]) if "edge_width" in posterior else edge_width_true
        pdf = bounded_smooth_mass_unnorm_np(
            m_grid, m_min_draw, m_max_draw, edge_width_draw, edge_pdf_value
        )
        pdf /= np.trapz(pdf, m_grid)
        pdf_draws.append(pdf)
    return np.array(pdf_draws)

def median_coeffs(posterior, order):
    coeff_draws = get_coeff_draws(posterior, order)
    return np.median(coeff_draws, axis=0)

def lambda_tilde_np(m1, m2, lambda1, lambda2):
    return (16.0 / 13.0) * (
        (m1 + 12.0 * m2) * m1**4 * lambda1
        + (m2 + 12.0 * m1) * m2**4 * lambda2
    ) / (m1 + m2) ** 5

def divergence_count(idata):
    if "diverging" not in idata.sample_stats:
        return np.nan
    return int(np.asarray(idata.sample_stats["diverging"]).sum())

## EOS posterior comparison



In [ ]:
m_plot = np.linspace(fit_min, fit_max, 300)
loglambda_truth = np.polyval(eos_coeffs_fit, m_plot)
lambda_truth = np.exp(loglambda_truth)

fig, ax = plt.subplots(figsize=(7.5, 4.8))

for label, result in all_results.items():
    loglambda_draws = eos_loglambda_draws(result["posterior"], result["order"], m_plot)
    lambda_draws = np.exp(loglambda_draws)
    q05, q50, q95 = result_band(lambda_draws)
    ax.fill_between(m_plot, q05, q95, alpha=0.18)
    ax.plot(m_plot, q50, lw=1.8, label=label)

ax.plot(m_plot, lambda_truth, color="black", lw=2.2, label="Injected truth")
ax.set_yscale("log")
ax.set_ylim(0.1, 1e5)
ax.set_xlabel(r"Mass $m\,[M_\odot]$")
ax.set_ylabel(r"$\Lambda(m)$")
ax.set_title("EOS posterior comparison")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

path = figures_dir / f"eos_misspec_comparison__{comparison_tag}.png"
fig.savefig(path, dpi=200)
plt.show()
print(f"Saved EOS misspecification comparison to {path}")

### What to notice

The matched model should usually recover the injected EOS curve within its posterior band.

The linear model may fail to reproduce curvature in $\log\Lambda(m)$. If the data are informative enough, this can produce a biased EOS posterior or a worse likelihood score.

However, if the detected events are few or the tidal measurements are broad, the linear model may still look acceptable. This does not mean the linear model is physically correct. It means the data are not strong enough to reject it in this realization.

The high-order model may produce a wider posterior band. This reflects the extra freedom allowed by the model. More flexibility is not automatically better: it can also increase prior dependence and make the inference less stable.

## Function-space EOS bias

This plot shows the difference between the inferred EOS curves and the true curve in log space,

$$
\Delta\log\Lambda(m)
=
\log\Lambda_{\rm inferred}(m)
-
\log\Lambda_{\rm true}(m).
$$

This is often more informative than comparing $\Lambda(m)$ directly, because $\Lambda$ spans orders of magnitude.

If

$$
\Delta\log\Lambda(m)=0,
$$

the inferred curve agrees with the true curve at that mass.

A positive value means the model predicts a larger tidal deformability than the truth. A negative value means it predicts a smaller tidal deformability.

Questions to answer:

1. Is the bias consistent with zero over the mass range populated by the events?

2. Is the bias largest near the edges of the mass range?

3. Does the underfit model show a coherent bias pattern?

4. Does the over-flexible model reduce bias at the cost of larger uncertainty?

This plot helps separate two effects:

- statistical uncertainty, represented by the width of the posterior band;
- systematic model bias, represented by a posterior band shifted away from zero.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))

bias_rows = []

for label, result in all_results.items():
    loglambda_draws = eos_loglambda_draws(result["posterior"], result["order"], m_plot)
    delta = loglambda_draws - loglambda_truth[None, :]
    q05, q50, q95 = result_band(delta)
    ax.fill_between(m_plot, q05, q95, alpha=0.18)
    ax.plot(m_plot, q50, lw=1.8, label=label)

    bias_rows.append({
        "model": label,
        "eos_order": result["order"],
        "mean_abs_delta_loglambda": float(np.mean(np.abs(q50))),
        "max_abs_delta_loglambda": float(np.max(np.abs(q50))),
    })

ax.axhline(0.0, color="black", lw=1.2)
ax.set_xlabel(r"Mass $m\,[M_\odot]$")
ax.set_ylabel(r"$\Delta\log\Lambda(m)$")
ax.set_title("EOS function-space bias")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

path = figures_dir / f"eos_misspec_bias__{comparison_tag}.png"
fig.savefig(path, dpi=200)
plt.show()
print(f"Saved EOS bias comparison to {path}")

## Mass-distribution comparison

The EOS model and the mass population are inferred jointly.

This means EOS misspecification can, in principle, affect the inferred mass distribution.

For example, if a wrong EOS model predicts tidal deformabilities that are too large or too small at a given mass, the inference may partially compensate by shifting the mass distribution.

This plot compares the inferred mass distributions,

$$
p(m|\lambda),
$$

for the different EOS models.

Questions to answer:

1. Does changing the EOS model noticeably shift the inferred mass range?

2. Are the mass-population parameters robust to EOS misspecification?

3. Does an underfit EOS model compensate by preferring different masses?

This is important because hierarchical inference couples all parts of the model. A mistake in one component can propagate into another.

In [ ]:
m_mass = np.linspace(fit_min, fit_max, 400)

pdf_true = bounded_smooth_mass_unnorm_np(
    m_mass, m_min_true, m_max_true, edge_width_true, edge_pdf_value
)
pdf_true /= np.trapz(pdf_true, m_mass)

fig, ax = plt.subplots(figsize=(7.5, 4.8))

for label, result in all_results.items():
    pdf_draws = mass_pdf_draws_from_posterior(result["posterior"], m_mass)
    q05, q50, q95 = result_band(pdf_draws)
    ax.fill_between(m_mass, q05, q95, alpha=0.18)
    ax.plot(m_mass, q50, lw=1.8, label=label)

ax.plot(m_mass, pdf_true, color="black", lw=2.2, label="Injected mass PDF")
ax.axvline(m_min_true, color="black", ls="--", lw=1.0, alpha=0.7)
ax.axvline(m_max_true, color="black", ls="--", lw=1.0, alpha=0.7)
ax.set_xlabel(r"Mass $m\,[M_\odot]$")
ax.set_ylabel(r"$p(m)$")
ax.set_title("Mass posterior under EOS model changes")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

path = figures_dir / f"mass_misspec_comparison__{comparison_tag}.png"
fig.savefig(path, dpi=200)
plt.show()
print(f"Saved mass misspecification comparison to {path}")

## Tidal residuals versus chirp mass

This plot compares the predicted tidal parameter to the true injected value for each detected event.

For each event and each inference model, we compute the median EOS prediction,

$$
\tilde{\Lambda}_{\rm pred}
=
\tilde{\Lambda}_{\rm EOS}^{(K)}
(m_1,m_2;\phi_{\rm EOS,med}^{(K)}),
$$

and compare it with the injected value,

$$
\tilde{\Lambda}_{\rm true}.
$$

The residual is plotted as

$$
\Delta\log\tilde{\Lambda}
=
\log\tilde{\Lambda}_{\rm pred}
-
\log\tilde{\Lambda}_{\rm true}.
$$

This diagnostic is event-based rather than function-based.

It asks:

> For the actual detected binaries, how wrong is the model prediction of the leading tidal parameter?

This matters because the EOS can be biased in a mass range where there are no events, but that bias may not affect the likelihood much.

Questions to answer:

1. Are the residuals centered near zero?

2. Does the underfit model show a systematic trend with chirp mass?

3. Are the residuals small compared with the PE uncertainty?

4. Do different EOS models make practically indistinguishable tidal predictions for the detected events?

The last question is crucial. If the predicted $\tilde{\Lambda}$ values are very similar for the detected events, then the data may not be able to distinguish the models even if their EOS curves differ elsewhere.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))

for label, result in all_results.items():
    coeff_med = median_coeffs(result["posterior"], result["order"])
    lambda1_med = np.exp(np.polyval(coeff_med, true_m1))
    lambda2_med = np.exp(np.polyval(coeff_med, true_m2))
    lambda_tilde_med = lambda_tilde_np(true_m1, true_m2, lambda1_med, lambda2_med)
    residual = (obs_log_lambdatilde - np.log(lambda_tilde_med)) / sigma_log_lambdatilde
    ax.scatter(true_mchirp, residual, s=18, alpha=0.55, label=label)

ax.axhline(0.0, color="black", lw=1.2)
ax.set_xlabel(r"Source-frame chirp mass $\mathcal{M}\,[M_\odot]$")
ax.set_ylabel(r"$(\log	ilde\Lambda_\mathrm{obs}-\log	ilde\Lambda_\mathrm{model})/\sigma$")
ax.set_title("Tidal residuals from posterior median EOS")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

path = figures_dir / f"tidal_residuals_vs_mass__{comparison_tag}.png"
fig.savefig(path, dpi=200)
plt.show()
print(f"Saved tidal residual comparison to {path}")

## Summary table

The summary table collects numerical diagnostics for each EOS model.

The most useful quantities are:

- the median hierarchical log likelihood;
- the DIC-like score;
- the mean and maximum absolute bias in $\log\Lambda(m)$;
- the mean and maximum event-level tidal residuals.

These diagnostics summarize different aspects of model performance.

A model with a better likelihood fits the detected events better.

A model with smaller function-space bias reproduces the true EOS curve better.

A model with smaller event-level residuals predicts the observed tidal parameters better.

These are related but not identical goals.

In real data we do not know the true EOS curve, so function-space bias cannot be computed. It is available here only because this is a controlled simulation.

In real analyses, we would rely on posterior predictive checks, model-comparison statistics, evidences, or information criteria.

In [ ]:
summary_rows = []
for row in bias_rows:
    label = row["model"]
    result = all_results[label]
    posterior = result["posterior"]

    summary_rows.append({
        "model": label,
        "eos_order": result["order"],
        "divergences": divergence_count(result["idata"]),
        "median_log_likelihood": float(np.median(posterior["log_likelihood_value"])),
        "median_log_likelihood_variance": float(np.median(posterior["log_likelihood_variance"])),
        "mean_abs_delta_loglambda": row["mean_abs_delta_loglambda"],
        "max_abs_delta_loglambda": row["max_abs_delta_loglambda"],
    })

comparison_summary = pd.DataFrame(summary_rows)
summary_path = results_dir / f"eos_misspec_summary__{comparison_tag}.csv"
comparison_summary.to_csv(summary_path, index=False)
print(f"Saved EOS misspecification summary to {summary_path}")
comparison_summary

## Model-comparison diagnostics

The DIC-like quantities computed below are posterior diagnostic scores based on the sampled hierarchical log likelihood.

They are not exact Bayesian evidences.

The basic idea is to compare models using both goodness of fit and effective model complexity.

A simple likelihood comparison can favor more flexible models because they have more freedom. A complexity-penalized score attempts to account for this.

The deviance is

$$
D(\lambda)
=
-2\log\mathcal{L}(\lambda).
$$

A DIC-like score has the schematic form

$$
{\rm DIC}
=
D(\bar{\lambda})
+
2p_{\rm eff},
$$

where $p_{\rm eff}$ is an estimate of the effective number of parameters.

In this notebook we use a variance-based diagnostic computed from the posterior distribution of the log likelihood. It should be interpreted qualitatively.

Questions to answer:

1. Which model has the best median log likelihood?

2. Which model has the best DIC-like score?

3. Do the differences look large or small?

4. Do the numerical diagnostics agree with the EOS posterior plots?

5. Would you confidently claim that one EOS model is preferred?

A very important conclusion is that small differences in DIC-like score or median log likelihood do not necessarily imply decisive model selection.

If the models are practically indistinguishable over the mass range probed by the detected events, the metrics may not strongly prefer the true model.

In [ ]:
# Model-comparison diagnostics.
#
# These are not Bayesian evidences. They are posterior diagnostic scores based on
# the sampled hierarchical log likelihood stored by the model.
#
# Definitions:
#   D(theta) = -2 log L(theta)
#   p_DIC_var = Var[D(theta)] / 2
#   DIC_var = mean[D(theta)] + p_DIC_var
#
# Lower DIC_var is better. Differences should be interpreted cautiously:
# the likelihood includes the selection correction and uses Monte Carlo estimates.

metric_rows = []

for label, result in all_results.items():
    posterior = result["posterior"]

    logL = np.asarray(posterior["log_likelihood_value"], dtype=float)
    logL_var = np.asarray(posterior["log_likelihood_variance"], dtype=float)
    log_xi_samples = np.asarray(posterior["log_xi"], dtype=float)

    finite = np.isfinite(logL)
    if not np.any(finite):
        raise RuntimeError(f"No finite log-likelihood samples for model {label}")

    logL = logL[finite]
    deviance = -2.0 * logL

    mean_deviance = np.mean(deviance)
    median_deviance = np.median(deviance)
    p_dic_var = 0.5 * np.var(deviance, ddof=1)
    dic_var = mean_deviance + p_dic_var

    metric_rows.append({
        "model": label,
        "eos_order": result["order"],
        "n_posterior_samples": len(logL),
        "divergences": divergence_count(result["idata"]),

        # Likelihood diagnostics
        "median_logL": np.median(logL),
        "q05_logL": np.quantile(logL, 0.05),
        "q95_logL": np.quantile(logL, 0.95),
        "mean_deviance": mean_deviance,
        "median_deviance": median_deviance,

        # DIC-like diagnostic
        "p_dic_var": p_dic_var,
        "dic_var": dic_var,

        # Selection / MC diagnostics
        "median_log_xi": np.median(log_xi_samples),
        "median_log_likelihood_variance": np.median(logL_var),
        "q95_log_likelihood_variance": np.quantile(logL_var, 0.95),
    })

model_metrics = pd.DataFrame(metric_rows)

# Differences relative to the matched model.
matched_label = f"Matched poly {truth_eos_order}"
if matched_label not in set(model_metrics["model"]):
    raise RuntimeError(f"Could not find matched reference model: {matched_label}")

matched_row = model_metrics.loc[model_metrics["model"] == matched_label].iloc[0]

model_metrics["delta_median_logL_vs_matched"] = (
    model_metrics["median_logL"] - matched_row["median_logL"]
)
model_metrics["delta_dic_var_vs_matched"] = (
    model_metrics["dic_var"] - matched_row["dic_var"]
)
model_metrics["delta_mean_deviance_vs_matched"] = (
    model_metrics["mean_deviance"] - matched_row["mean_deviance"]
)

# Add function-space bias metrics already computed in bias_rows.
bias_metrics = pd.DataFrame(bias_rows)
model_metrics = model_metrics.merge(
    bias_metrics[[
        "model",
        "mean_abs_delta_loglambda",
        "max_abs_delta_loglambda",
    ]],
    on="model",
    how="left",
)

# Sort by DIC-like score.
model_metrics = model_metrics.sort_values("dic_var").reset_index(drop=True)

metrics_path = results_dir / f"eos_misspec_model_metrics__{comparison_tag}.csv"
model_metrics.to_csv(metrics_path, index=False)

print(f"Saved EOS model-comparison metrics to {metrics_path}")
display(model_metrics)


# Automatic text conclusions.
best_by_dic = model_metrics.iloc[0]
best_by_logL = model_metrics.loc[model_metrics["median_logL"].idxmax()]

print("\nModel-comparison conclusions")
print("----------------------------")
print(
    f"Best DIC-like score: {best_by_dic['model']} "
    f"(DIC_var = {best_by_dic['dic_var']:.2f})."
)
print(
    f"Best median log likelihood: {best_by_logL['model']} "
    f"(median logL = {best_by_logL['median_logL']:.2f})."
)

for _, row in model_metrics.iterrows():
    if row["model"] == matched_label:
        continue

    print()
    print(f"{row['model']} compared with {matched_label}:")
    print(f"  Δ median logL = {row['delta_median_logL_vs_matched']:.2f}")
    print(f"  Δ DIC_var     = {row['delta_dic_var_vs_matched']:.2f}")
    print(f"  mean |Δ log Lambda| = {row['mean_abs_delta_loglambda']:.3f}")
    print(f"  max  |Δ log Lambda| = {row['max_abs_delta_loglambda']:.3f}")

    if abs(row["delta_median_logL_vs_matched"]) < 2.0 and abs(row["delta_dic_var_vs_matched"]) < 5.0:
        print("  Interpretation: practically indistinguishable from the matched model in this run.")
    elif row["delta_median_logL_vs_matched"] < -2.0 or row["delta_dic_var_vs_matched"] > 5.0:
        print("  Interpretation: disfavored relative to the matched model by these diagnostics.")
    else:
        print("  Interpretation: weak or ambiguous preference; inspect residual and bias plots.")